Library

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import timm
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import cv2
from torch.utils.data import Dataset, DataLoader


MobileNetV2

In [ ]:
# CONFIG (Update these to your Split folders)
TRAIN_PATH = "/content/Static_image_final/train"
VAL_PATH   = "/content/Static_image_final/val"
TEST_PATH  = "/content/Static_image_final/test"

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-3
PATIENCE = 8

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

# SIMPLIFIED DATA LOADER (Flattens Front/Side folders)
class FlatASLDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        if not os.path.exists(root_dir):
            raise FileNotFoundError(f"error")

        self.classes = sorted([d for d in os.listdir(root_dir) if not d.startswith('.')])
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        self.samples = []

        for cls in self.classes:
            class_path = os.path.join(root_dir, cls)
            for view in ['Front', 'Side']:
                view_path = os.path.join(class_path, view)
                if os.path.exists(view_path):
                    for img_name in os.listdir(view_path):
                        # Filter for images and ignore hidden files like ._sign_ or .DS_Store
                        if img_name.lower().endswith(('.jpg', '.png', '.jpeg')) and not img_name.startswith('.'):
                            full_path = os.path.join(view_path, img_name)
                            # Double-check existence to prevent the PIL error
                            if os.path.exists(full_path):
                                self.samples.append((full_path, self.class_to_idx[cls]))

        if len(self.samples) == 0:
            print(f"No images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception as e:
            raise e

# DATA LOADERS 
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = FlatASLDataset(TRAIN_PATH, transform=train_transform)
val_dataset   = FlatASLDataset(VAL_PATH, transform=val_transform)
test_dataset  = FlatASLDataset(TEST_PATH, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

# MODEL (MobileNetV2)
model = timm.create_model("mobilenetv2_100", pretrained=True)

# Freeze backbone for initial training
for param in model.parameters():
    param.requires_grad = False

# Replace classifier for your specific number of classes
in_features = model.get_classifier().in_features
model.classifier = nn.Sequential(
    nn.BatchNorm1d(in_features),
    nn.Dropout(0.2),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

# LOSS + OPTIMIZER
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# TRAINING LOOP
best_val_acc = 0
patience_counter = 0
train_acc_history, val_acc_history = [], []

for epoch in range(EPOCHS):
    model.train()
    train_correct, train_total = 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total

    # Validation Phase
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    scheduler.step()

    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)
    print(f"Epoch {epoch+1}: Train Acc = {train_acc:.4f} | Val Acc = {val_acc:.4f}")

    # Unfreeze deeper layers for fine-tuning at Epoch 4
    if epoch == 3:
        for name, param in model.named_parameters():
            if "blocks.14" in name or "blocks.15" in name:
                param.requires_grad = True
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    # Early Stopping & Checkpointing
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_asl_model.pth")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:

            break

# FINAL EVALUATION
model.load_state_dict(torch.load("best_asl_model.pth"))
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_true.extend(labels.cpu().numpy())

test_acc = np.mean(np.array(all_preds) == np.array(all_true))
print(f"\nFinal Test Accuracy: {test_acc:.4f}")
print("\nClassification Report:\n", classification_report(all_true, all_preds, target_names=class_names))

# Confusion Matrix Visualization
cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.title("Test Confusion Matrix")
plt.show()

EffecientNet_b0

In [ ]:
# CONFIG
TEST_PATH = "/content/Static_image_final/test"
TRAIN_PATH = "/content/Static_image_final/train"
VAL_PATH = "/content/Static_image_final/val"
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-3
PATIENCE = 8

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

# DATA LOADER 
class FlatASLDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        if not os.path.exists(root_dir):
            raise FileNotFoundError(f"error")

        self.classes = sorted([d for d in os.listdir(root_dir) if not d.startswith('.')])
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        self.samples = []

        for cls in self.classes:
            class_path = os.path.join(root_dir, cls)
            for view in ['Front', 'Side']:
                view_path = os.path.join(class_path, view)
                if os.path.exists(view_path):
                    for img_name in os.listdir(view_path):
                        # Filter for images and ignore hidden files like ._sign_ or .DS_Store
                        if img_name.lower().endswith(('.jpg', '.png', '.jpeg')) and not img_name.startswith('.'):
                            full_path = os.path.join(view_path, img_name)
                            # Double-check existence to prevent the PIL error
                            if os.path.exists(full_path):
                                self.samples.append((full_path, self.class_to_idx[cls]))

        if len(self.samples) == 0:
            print(f"error")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception as e:
            raise e


# TRANSFORMS
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = FlatASLDataset(TRAIN_PATH, transform=train_transform)
val_dataset   = FlatASLDataset(VAL_PATH, transform=val_transform)
test_dataset  = FlatASLDataset(TEST_PATH, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Classes: {class_names} | Total Train: {len(train_dataset)}")

# MODEL (EfficientNet-B0)
model = timm.create_model("efficientnet_b0", pretrained=True)

# Freeze most layers
for param in model.parameters():
    param.requires_grad = False

# Replace classifier
in_features = model.get_classifier().in_features
model.classifier = nn.Sequential(
    nn.BatchNorm1d(in_features),
    nn.Dropout(0.2),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

# LOSS + OPTIMIZER
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# TRAINING LOOP
best_val_acc = 0
patience_counter = 0

train_acc_history = []
val_acc_history = []

for epoch in range(EPOCHS):

    #TRAIN 
    model.train()
    train_correct, train_total = 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        preds = outputs.argmax(1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total

    #VALIDATION 
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    scheduler.step()

    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)

    print(f"Epoch {epoch+1}: Train Acc = {train_acc:.4f} | Val Acc = {val_acc:.4f}")

    # UNFREEZE LAST BLOCK 
    if epoch == 5:
        print(" Unfreezing last layers...")
        for name, param in model.named_parameters():
            if "blocks.6" in name or "blocks.7" in name:
                param.requires_grad = True

    # EARLY STOP 
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_efficientnet.pth")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(" Early stopping")
            break

# PLOT ACCURACY
plt.figure(figsize=(10,5))
plt.plot(train_acc_history, label="Train")
plt.plot(val_acc_history, label="Validation")
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# TEST EVALUATION
model.load_state_dict(torch.load("best_efficientnet.pth"))
model.eval()

test_correct, test_total = 0, 0
all_preds, all_true = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(1)

        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

test_acc = test_correct / test_total

print("\n Test Accuracy:", test_acc)

# SAVE MODEL
model_name = f"efficientnet_{test_acc:.4f}.pth"
torch.save(model.state_dict(), model_name)
print(f" Model saved as: {model_name}")

# REPORT + CONFUSION MATRIX
print("\nClassification Report:\n")
print(classification_report(all_true, all_preds, target_names=class_names))

cm = confusion_matrix(all_true, all_preds)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names)
plt.title("Test Confusion Matrix")
plt.show()

# SUMMARY
print("\n FINAL RESULTS")
print(f"Final Training Accuracy: {train_acc_history[-1]:.4f}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

ResNet-18

In [ ]:
# CONFIG
TEST_PATH = "/content/Static_image_final/test"
TRAIN_PATH = "/content/Static_image_final/train"
VAL_PATH = "/content/Static_image_final/val"
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-3
PATIENCE = 8

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

# DATA LOADER 
class FlatASLDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        if not os.path.exists(root_dir):
            raise FileNotFoundError(f"error")

        self.classes = sorted([d for d in os.listdir(root_dir) if not d.startswith('.')])
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        self.samples = []

        for cls in self.classes:
            class_path = os.path.join(root_dir, cls)
            for view in ['Front', 'Side']:
                view_path = os.path.join(class_path, view)
                if os.path.exists(view_path):
                    for img_name in os.listdir(view_path):
                        # Filter for images and ignore hidden files like ._sign_ or .DS_Store
                        if img_name.lower().endswith(('.jpg', '.png', '.jpeg')) and not img_name.startswith('.'):
                            full_path = os.path.join(view_path, img_name)
                            # Double-check existence to prevent the PIL error
                            if os.path.exists(full_path):
                                self.samples.append((full_path, self.class_to_idx[cls]))

        if len(self.samples) == 0:
            print(f"Warning: No images found in {root_dir}. Check folder structure.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception as e:    
            raise e



class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Classes: {class_names} | Total Train: {len(train_dataset)}")

# Augmentation
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),

])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = FlatASLDataset(TRAIN_PATH, transform=train_transform)
val_dataset   = FlatASLDataset(VAL_PATH, transform=val_transform)
test_dataset  = FlatASLDataset(TEST_PATH, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# MODEL (ResNet18)
model = timm.create_model("resnet18", pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace classifier
in_features = model.get_classifier().in_features
model.fc = nn.Sequential(
    nn.BatchNorm1d(in_features),
    nn.Dropout(0.2),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

# LOSS + OPTIMIZER
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# TRAINING LOOP
best_val_acc = 0
patience_counter = 0

train_acc_history = []
val_acc_history = []

for epoch in range(EPOCHS):

    #TRAIN 
    model.train()
    train_correct, train_total = 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        preds = outputs.argmax(1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total

    # VALIDATION 
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    scheduler.step()

    # store history
    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)

    print(f"Epoch {epoch+1}: Train Acc = {train_acc:.4f} | Val Acc = {val_acc:.4f}")

    # UNFREEZE LAST BLOCK
    if epoch == 5:
        print("Unfreezing last layer...")
        for name, param in model.named_parameters():
            if "layer4" in name:
                param.requires_grad = True

    #EARLY STOP
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_resnet18.pth")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping")
            break

# PLOT ACCURACY CURVE
plt.figure(figsize=(10,5))
plt.plot(train_acc_history, label="Train Accuracy")
plt.plot(val_acc_history, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

# TEST EVALUATION
model.load_state_dict(torch.load("best_resnet18.pth"))
model.eval()

test_correct, test_total = 0, 0
all_preds, all_true = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(1)

        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

test_acc = test_correct / test_total

print("\n Test Accuracy:", test_acc)


# SAVE MODEL WITH ACCURACY
model_name = f"resnet18_{test_acc:.4f}.pth"
torch.save(model.state_dict(), model_name)

print(f" Model saved as: {model_name}")

# CONFUSION MATRIX + REPORT
print("\nClassification Report:\n")
print(classification_report(all_true, all_preds, target_names=class_names))

cm = confusion_matrix(all_true, all_preds)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names)
plt.title("Test Confusion Matrix")
plt.show()

# FINAL SUMMARY
print("\n FINAL RESULTS")
print(f"Final Training Accuracy: {train_acc_history[-1]:.4f}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")